<a href="https://colab.research.google.com/github/natharzu/Practical-Data-Science/blob/main/dataviz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Topics vs. AI Perception in News Media

The visualization is based on a dataset of news articles mentioning artificial intelligence, enriched with metadata extracted through automated text classification. Each article includes:

AI sentiment:

Helper — AI framed as beneficial

Neutral — descriptive or balanced coverage

Threat — AI framed as harmful or risky

AI topic category (derived from NIST taxonomy):
Examples include content creation, cybersecurity, automation, research, monitoring, prediction, etc.

Publication year

To improve interpretability, similar categories were grouped (e.g., “content creation”, “content generation”, “content synthesis” → Content), and rare categories were combined into Other (rare).

The final dataset contains several thousand articles across multiple years and dozens of AI‑related topics.



In [129]:
#2 README.md
%%writefile README.md


## Dataset
Includes:
- AI sentiment labels
- NIST-based topic categories
- Publication year
Rare categories grouped as "Other (rare)".

https://huggingface.co/datasets/kkChimmy/REALM

## Design Choices
- Heatmap for multi-category comparison
- Log scale for skewed data
- Light Greens palette
- Unified style: Arial font, centered titles, light grid

## Structure
notebooks/   Data processing and visualization
images/      Exported heatmap
report.pdf   Final assignment report
README.md    Project overview



Overwriting README.md


In [49]:
#=====================================
#1 Загрузка датасета и первичная очистка
#=====================================

import pandas as pd
from datasets import load_dataset

# Загрузка
ds = load_dataset("kkChimmy/REALM")
df = ds['train'].to_pandas()

# Размер и первые строки
print("Shape:", df.shape)
display(df.head())

# Типы данных
print("\nDtypes:")
print(df.dtypes)

# Список колонок
print("\nColumns:")
print(df.columns.tolist())

# Преобразование publishedAt → год
df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
df['year'] = df['publishedAt'].dt.year

# Проверяем, что получилось
print("Распределение по годам:")
print(df['year'].value_counts(dropna=False).sort_index())

print("\nКоличество строк без даты (NaT):", df['year'].isna().sum())

# Примеры строк без даты (для отладки)
print("\nПримеры строк, где год не определён:")
print(df[df['year'].isna()][['title', 'source', 'publishedAt']].head(5))

# Привести категориальные поля к category
cat_cols = ["category", "category_nist", "subreddit"]
for col in cat_cols:
    df[col] = df[col].astype("category")

# Проверка top_comments на содержимое
#print(df["top_comments"].head(10))
#print(df["top_comments"].notnull().sum())

# Удаляем колонку top_comments, так как она не содержит данные
df = df.drop(columns=["top_comments"])

# Проверяем результат удаления
df.columns





Shape: (93259, 16)


,source,author,title,description,url,urlToImage,publishedAt,content,category_nist,category,id,subreddit,score,num_comments,created_time,top_comments
0,news,Apundir,101° - Machine Learning The complete Math Guid...,Machine Learning is rapidly changing the world...,https://www.hotukdeals.com/deals/kindle-editio...,https://images.hotukdeals.com/threads/raw/defa...,2021-08-23T12:48:28Z,hotukdeals.com - The Largest Deal Community in...,Content Synthesis/Digital Assistance/Informati...,"Computer and Mathematical/Education, Training,...",None,None,NaN,NaN,NaT,None
1,news,Wall Street Reporter,"Next Super Stocks on the Move: Logiq, Reliq He...","NEW YORK, Sept. 02, 2021 (GLOBE NEWSWIRE) -- W...",https://finance.yahoo.com/news/next-super-stoc...,https://s.yimg.com/cv/apiv2/social/images/yaho...,2021-09-02T13:27:00Z,"NEW YORK, Sept. 02, 2021 (GLOBE NEWSWIRE) -- W...",Content Synthesis/Decision Making/Recommendation,Healthcare Practitioners and Support/Managemen...,None,None,NaN,NaN,NaT,None
2,news,,"How, where and why telco is using enterprise o...","According to ""The State of Enterprise Open Sou...",https://www.redhat.com/en/blog/how-where-and-w...,https://www.redhat.com/cms/managed-files/style...,2021-08-23T04:00:00Z,While the telecommunications industry is famil...,Unknown,Management/Computer and Mathematical,None,None,NaN,NaN,NaT,None
3,news,"Dario D'Amico, Dario D'Amico",Visualize and animate flow in MapView with a c...,Learn how to animate streamlines using WebGL a...,https://www.esri.com/arcgis-blog/products/js-a...,https://www.esri.com/arcgis-blog/wp-content/up...,2021-09-01T00:30:08Z,IntroductionThis article discusses visualizing...,Content Creation/Content Synthesis,"Computer and Mathematical/Life, Physical, and ...",None,None,NaN,NaN,NaT,None
4,news,Michelle Horton,Deciphering Ancient Texts with AI,Using traditional machine learning methods and...,https://developer.nvidia.com/blog/deciphering-...,https://developer-blogs.nvidia.com/wp-content/...,2021-08-12T17:57:09Z,Using traditional machine learning methods and...,Process Automation/Content Creation,Unknown,None,None,NaN,NaN,NaT,None



Dtypes:
source                   object
author                   object
title                    object
description              object
url                      object
urlToImage               object
publishedAt              object
content                  object
category_nist            object
category                 object
id                       object
subreddit                object
score                   float64
num_comments            float64
created_time     datetime64[ns]
top_comments             object
dtype: object

Columns:
['source', 'author', 'title', 'description', 'url', 'urlToImage', 'publishedAt', 'content', 'category_nist', 'category', 'id', 'subreddit', 'score', 'num_comments', 'created_time', 'top_comments']
Распределение по годам:
year
2020.0     1108
2021.0     2092
2022.0     3123
2023.0    28253
2024.0    44173
NaN       14510
Name: count, dtype: int64

Количество строк без даты (NaT): 14510

Примеры строк, где год не определён:
                             

Index(['source', 'author', 'title', 'description', 'url', 'urlToImage',
       'publishedAt', 'content', 'category_nist', 'category', 'id',
       'subreddit', 'score', 'num_comments', 'created_time', 'year'],
      dtype='object')

In [78]:
#=================================
#2 Очистка датасета
#=================================


print("Исходный размер:", df.shape)

# Удаляем полные дубликаты (если вдруг остались)
df = df.drop_duplicates(subset=['url', 'title'])

# Обрабатываем даты
df['publishedAt'] = pd.to_datetime(df['publishedAt'], errors='coerce')
df['year'] = df['publishedAt'].dt.year

# Отсекаем аномальные годы
df = df[df['year'].between(2020, 2025)]

# Критические пропуски — удаляем строки без ключевых полей
df = df.dropna(subset=["category", "category_nist", "publishedAt"])


# Приводим категории к нижнему регистру и убираем пробелы
df["category"] = df["category"].str.strip().str.lower()
df["category_nist"] = df["category_nist"].str.strip().str.lower()

# Удаляем 'мусорные' категории
bad_categories = {"unknown", "error", "none", "null", "undefined", ""}
df = df[~df["category"].isin(bad_categories)]
df = df[~df["category_nist"].isin(bad_categories)]


# Разбиваем множественные категории (детальный heatmap)
df_exploded = (
    df.assign(category_nist=df["category_nist"].str.split("/"))
      .explode("category_nist")
)

df_exploded["category_nist"] = df_exploded["category_nist"].str.strip()

# Финальный размер после очистки
print("Размер после очистки:", df.shape)
print("Пропуски в ключевых колонках после очистки:\n", df[['category', 'category_nist', 'year']].isna().sum())

# Теперь можно аггрегировать и строить график

Исходный размер: (35414, 20)
Размер после очистки: (35414, 20)
Пропуски в ключевых колонках после очистки:
 category         0
category_nist    0
year             0
dtype: int64


In [114]:
#=================================
#3 Анализ и визуализация-3
#=================================

import pandas as pd

# -----------------------------
# 1. Словари ключевых слов
# -----------------------------
threat_words = [
    "risk", "threat", "danger", "replace", "job loss", "automation",
    "unemployment", "harm", "fear", "concern", "replace workers",
    "layoffs", "job displacement", "crisis"
]

helper_words = [
    "assist", "help", "enhance", "augment", "boost", "support",
    "productivity", "improve", "benefit", "collaborate", "empower",
    "increase efficiency", "streamline"
]

# -----------------------------
# 2. Функция подсчёта ключевых слов
# -----------------------------
def count_keywords(text, keywords):
    if pd.isna(text):
        return 0
    text = text.lower()
    return sum(1 for w in keywords if w in text)

# -----------------------------
# 3. Подсчёт "угроза" и "помощник"
# -----------------------------
df["threat_score"] = df["content"].apply(lambda x: count_keywords(x, threat_words))
df["helper_score"] = df["content"].apply(lambda x: count_keywords(x, helper_words))

# -----------------------------
# 4. Классификация новости
# -----------------------------
def classify_sentiment(row):
    if row["threat_score"] > row["helper_score"]:
        return "threat"
    elif row["helper_score"] > row["threat_score"]:
        return "helper"
    else:
        return "neutral"

df["ai_sentiment"] = df.apply(classify_sentiment, axis=1)

# -----------------------------
# 5. Быстрая проверка результата
# -----------------------------
print(df["ai_sentiment"].value_counts())
df[["title", "ai_sentiment", "threat_score", "helper_score"]].head(10)



ai_sentiment
helper     25349
neutral     5816
threat      4249
Name: count, dtype: int64


,title,ai_sentiment,threat_score,helper_score
0,101° - Machine Learning The complete Math Guid...,neutral,0,0
1,"Next Super Stocks on the Move: Logiq, Reliq He...",helper,0,1
3,Visualize and animate flow in MapView with a c...,helper,0,4
5,AnChain.AI Raises $10 Million and Wins SEC Con...,helper,2,3
8,Remark Holdings Announces Fiscal Second Quarte...,helper,1,3
9,Data Engineering Annotated Monthly – August 2021,helper,0,3
10,quantlet added to PyPI,helper,0,1
15,"Verint Acquires Conversocial, Salesforce Wants...",helper,0,3
16,How to Implement Artificial Intelligence in Ma...,helper,1,4
19,Cambridge Judge Business School opens courses ...,helper,1,4


In [113]:
# ============================================================
# DASHBOARD: AI perception in media
# 3 charts, unified style, English labels
# ============================================================

import plotly.express as px
import plotly.graph_objects as go
import numpy as np

# ------------------------------------------------------------
# 1. Data preparation
# ------------------------------------------------------------

df_exploded["category_main"] = df_exploded["category_nist"].str.split("/").str[0]

sentiment_colors = {
    "helper": "#4CAF50",
    "neutral": "#B0BEC5",
    "threat": "#EF5350"
}

sentiment_labels = {
    "helper": "Helper",
    "neutral": "Neutral",
    "threat": "Threat"
}

# ------------------------------------------------------------
# 2. Unified style
# ------------------------------------------------------------

unified_layout = dict(
    template="plotly_white",
    font=dict(family="Arial, sans-serif", size=14, color="#333"),
    margin=dict(l=80, r=80, t=80, b=80),
)

unified_xaxis = dict(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.05)",
    zeroline=False,
    tickfont=dict(size=13),
    title_font=dict(size=16)
)

unified_yaxis = dict(
    showgrid=True,
    gridcolor="rgba(0,0,0,0.05)",
    zeroline=False,
    tickfont=dict(size=13),
    title_font=dict(size=16),
    automargin=True
)

# ------------------------------------------------------------
# 3. Chart 1: News categories (horizontal bar chart)
# ------------------------------------------------------------

cat_counts = df_exploded["category_main"].value_counts()
top_n = 15
top_categories = cat_counts.head(top_n).index

df_exploded["category_main_top"] = df_exploded["category_main"].apply(
    lambda x: x if x in top_categories else "Other"
)

df_cat_top = (
    df_exploded
    .groupby(["category_main_top", "ai_sentiment"])
    .size()
    .reset_index(name="count")
)

fig_hbar = go.Figure()

fig_hbar.add_trace(
    go.Bar(
        y=df_cat_top["category_main_top"],
        x=df_cat_top["count"],
        orientation="h",
        marker_color=[sentiment_colors[s] for s in df_cat_top["ai_sentiment"]],
        customdata=df_cat_top[["ai_sentiment", "count"]],
        hovertemplate=
            "<b>%{y}</b><br>" +
            "Sentiment: <b>%{customdata[0]}</b><br>" +
            "News count: <b>%{customdata[1]}</b><br>" +
            "<extra></extra>"
    )
)

fig_hbar.update_layout(
    **unified_layout,
    title=dict(
        text="News Categories (TOP‑15 + Other)",
        x=0.5,
        font=dict(family="Arial Black", size=26, color="#222")
    )
)

fig_hbar.update_xaxes(title_text="News count", **unified_xaxis)
fig_hbar.update_yaxes(title_text="Category", **unified_yaxis)

fig_hbar.show()

# ------------------------------------------------------------
# 4. Chart 2: AI perception dynamics by year
# ------------------------------------------------------------

df_year = df_exploded.groupby(["year", "ai_sentiment"]).size().reset_index(name="count")

fig_year = go.Figure()

for sentiment in df_year["ai_sentiment"].unique():
    subset = df_year[df_year["ai_sentiment"] == sentiment]
    fig_year.add_trace(
        go.Scatter(
            x=subset["year"],
            y=subset["count"],
            mode="lines+markers",
            name=sentiment_labels[sentiment],
            line=dict(color=sentiment_colors[sentiment], width=3),
            customdata=subset["count"],
            hovertemplate=
                "<b>%{x}</b><br>" +
                "Sentiment: <b>" + sentiment_labels[sentiment] + "</b><br>" +
                "News count: <b>%{customdata}</b><br>" +
                "<extra></extra>"
        )
    )

fig_year.update_layout(
    **unified_layout,
    title=dict(
        text="AI Perception Dynamics by Year",
        x=0.5,
        font=dict(family="Arial Black", size=26, color="#222")
    )
)

fig_year.update_xaxes(title_text="Year", **unified_xaxis)
fig_year.update_yaxes(title_text="News count", **unified_yaxis)

fig_year.show()

# ------------------------------------------------------------
# 5. Chart 3: Heatmap (log scale + grouping + light palette)
# ------------------------------------------------------------

def group_category(cat):
    cat = cat.lower()
    if "content" in cat:
        return "Content"
    if "information retrieval" in cat or "search" in cat:
        return "Information retrieval"
    if "automation" in cat:
        return "Automation"
    if "monitor" in cat or "detect" in cat:
        return "Monitoring"
    if "research" in cat:
        return "Research"
    if "predict" in cat:
        return "Prediction"
    if "education" in cat:
        return "Education"
    if "ethical" in cat:
        return "Ethical considerations"
    if "cyber" in cat:
        return "Cybersecurity"
    return cat.capitalize()

df_exploded["category_grouped"] = df_exploded["category_main"].apply(group_category)

cat_counts = df_exploded["category_grouped"].value_counts()
top_cats = cat_counts.head(20).index

df_exploded["category_grouped_top"] = df_exploded["category_grouped"].apply(
    lambda x: x if x in top_cats else "Other (rare)"
)

pivot = df_exploded.pivot_table(
    index="category_grouped_top",
    columns="ai_sentiment",
    aggfunc="size",
    fill_value=0
)

z_log = np.log1p(pivot.values)

fig_heat = go.Figure(
    go.Heatmap(
        z=z_log,
        x=[sentiment_labels[c] for c in pivot.columns],
        y=pivot.index,
        colorscale="Greens",
        colorbar=dict(
            title="log(1 + count)",
            ticks="outside",
            tickfont=dict(size=12)
        ),
        customdata=pivot.values,
        hovertemplate=
            "<b>%{y}</b><br>" +
            "Sentiment: <b>%{x}</b><br>" +
            "News count: <b>%{customdata}</b><br>" +
            "<extra></extra>"
    )
)

fig_heat.update_layout(
    **unified_layout,
    title=dict(
        text="Topics vs AI Perception (log scale)",
        x=0.5,
        font=dict(family="Arial Black", size=26, color="#222")
    )
)

fig_heat.update_xaxes(title_text="Sentiment", **unified_xaxis)
fig_heat.update_yaxes(title_text="Topic", **unified_yaxis)

fig_heat.show()
